In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
def get_housing_data():
    data = fetch_california_housing(as_frame=True)
    X = data.data
    y = data.target
    return X, y


In [14]:
y.shape

(20640,)

In [6]:
def split_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    return X_train, X_test, y_train, y_test


In [9]:
class MiniBatchGDRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, learning_rate=0.01, epochs=100, batch_size=32):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None
        self.cost_history = []
        
    def fit(self, X, y):
        # Arrays mein convert karna zaroori hai
        X = np.array(X)
        y = np.array(y).reshape(-1) 
        
        m_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        self.cost_history = []
        
        # Training loop
        for epoch in range(self.epochs):
            # Data shuffling for randomness in batches
            indices = np.random.permutation(m_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # Mini-batch process
            for i in range(0, m_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]
                batch_m = X_batch.shape[0]
                
                # Hypothesis (Predictions)
                y_pred = np.dot(X_batch, self.weights) + self.bias
                
                # Error
                error = y_pred - y_batch
                
                # Gradients
                dw = (2 / batch_m) * np.dot(X_batch.T, error)
                db = (2 / batch_m) * np.sum(error)
                
                # Parameters Update
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db
                
            # Har epoch ke end par loss track karna
            y_pred_total = np.dot(X, self.weights) + self.bias
            mse_loss = np.mean((y_pred_total - y)**2)
            self.cost_history.append(mse_loss)
            
        return self
    
    def predict(self, X):
        X = np.array(X)
        return np.dot(X, self.weights) + self.bias

In [10]:
def create_ml_pipeline():
    pipeline = Pipeline([
        ('scaler', StandardScaler()), 
        # Tuning parameters for better convergence
        ('regressor', MiniBatchGDRegressor(learning_rate=0.01, epochs=150, batch_size=64))
    ])

    return pipeline


In [11]:
def evaluate_model(model_pipeline, X_test, y_test):
    y_pred = model_pipeline.predict(X_test)
    
    # Calculate Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"Mean Absolute Error (MAE) : {mae:.4f}")
    print(f"Mean Squared Error (MSE)  : {mse:.4f}")
    print(f"R² Score                  : {r2:.4f}\n")
    
    # Compare Top 10 Predictions
    # Using np.array for y_test just in case it's a pandas Series to avoid index mismatch
    y_test_arr = np.array(y_test)
    comparison_df = pd.DataFrame({
        'Actual ($100k)': np.round(y_test_arr[:10], 3),
        'Predicted ($100k)': np.round(y_pred[:10], 3),
        'Difference': np.round(abs(y_test_arr[:10] - y_pred[:10]), 3)
    })
    
    print("--- Actual vs Predicted (Top 10) ---")
    print(comparison_df.to_string(index=False))
    return y_pred

In [12]:
if __name__ == "__main__":
    # Step 1: Load Data
    X, y = get_housing_data()
    
    # Step 2: Split Data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Step 4: Initialize Pipeline
    model_pipeline = create_ml_pipeline()
    
    # Step 5: Model Training
    print("Model training started... (This might take a few seconds)")
    model_pipeline.fit(X_train, y_train)
    print("Model training completed!\n")
    
    # Step 6 & 7: Evaluate and Predict
    predictions = evaluate_model(model_pipeline, X_test, y_test)

Model training started... (This might take a few seconds)
Model training completed!

Mean Absolute Error (MAE) : 0.5330
Mean Squared Error (MSE)  : 0.5659
R² Score                  : 0.5681

--- Actual vs Predicted (Top 10) ---
 Actual ($100k)  Predicted ($100k)  Difference
          0.477              0.704       0.227
          0.458              1.760       1.302
          5.000              2.699       2.301
          2.186              2.848       0.662
          2.780              2.608       0.172
          1.587              2.008       0.421
          1.982              2.646       0.664
          1.575              2.161       0.586
          3.400              2.729       0.671
          4.466              3.925       0.541
